In [ ]:
import h5py
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    Dense,
    Dropout,
    Flatten,
    BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping

# DATASET FILES

Dataset_files = [
    "H-H1_GWOSC_4KHZ_R1-1187008867-32.hdf5",
    "H-H1_GWOSC_16KHZ_R1-1187008867-32.hdf5",
    "L-L1_GWOSC_4KHZ_R1-1187008867-32.hdf5",
    "L-L1_GWOSC_16KHZ_R1-1187008867-32.hdf5",
    "V-V1_GWOSC_4KHZ_R1-1187008867-32.hdf5",
    "V-V1_GWOSC_16KHZ_R1-1187008867-32.hdf5"
    "H-H1_GWOSC_4KHZ_R1-1187006835-4096.hdf5",
    "H-H1_GWOSC_16KHZ_R1-1187006835-4096.hdf5",
    "L-L1_GWOSC_4KHZ_R1-1187006835-4096.hdf5",
    "L-L1_GWOSC_16KHZ_R1-1187006835-4096.hdf5",
    "V-V1_GWOSC_4KHZ_R1-1187006835-4096.hdf5"
]

# LOAD STRAIN

def load_strain(file_path):
    with h5py.File(file_path, "r") as f:
        strain = f["strain"]["Strain"][:]
    return strain

# WINDOW CREATION

WINDOW_SIZE = 4096
STEP_SIZE = 2048

X = []
y = []

def generate_windows(signal, label):

    for i in range(
        0,
        len(signal) - WINDOW_SIZE,
        STEP_SIZE
    ):

        segment = signal[i:i+WINDOW_SIZE]

        X.append(segment)
        y.append(label)

# Positive samples
for file in positive_files:
    signal = load_strain(file)
    generate_windows(signal, 1)

# Negative samples
for file in negative_files:
    signal = load_strain(file)
    generate_windows(signal, 0)

X = np.array(X)
y = np.array(y)

print("Dataset Shape:", X.shape)

# NORMALIZATION

scaler = StandardScaler()

X = scaler.fit_transform(X)

# CNN expects:
# (samples, timesteps, channels)

X = X.reshape(
    X.shape[0],
    X.shape[1],
    1
)

print("CNN Input Shape:", X.shape)

# TRAIN TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# CNN MODEL

model = Sequential()

# Block 1
model.add(
    Conv1D(
        filters=32,
        kernel_size=7,
        activation='relu',
        input_shape=(
            X_train.shape[1],
            1
        )
    )
)

model.add(BatchNormalization())

model.add(
    MaxPooling1D(
        pool_size=2
    )
)

# Block 2
model.add(
    Conv1D(
        filters=64,
        kernel_size=5,
        activation='relu'
    )
)

model.add(BatchNormalization())

model.add(
    MaxPooling1D(
        pool_size=2
    )
)

# Block 3
model.add(
    Conv1D(
        filters=128,
        kernel_size=3,
        activation='relu'
    )
)

model.add(BatchNormalization())

model.add(
    MaxPooling1D(
        pool_size=2
    )
)

# Dense Layers
model.add(Flatten())

model.add(Dense(
    256,
    activation='relu'
))

model.add(
    Dropout(0.4)
)

model.add(Dense(
    128,
    activation='relu'
))

model.add(
    Dropout(0.3)
)

model.add(Dense(
    1,
    activation='sigmoid'
))

# COMPILE

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# EARLY STOPPING

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# TRAIN

history = model.fit(
    X_train,
    y_train,
    validation_split=0.20,
    epochs=100,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

# EVALUATION

loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("\nTest Accuracy:", accuracy)

# PREDICTIONS

predictions = model.predict(X_test)

predictions = (
    predictions > 0.5
).astype(int)

print(
    classification_report(
        y_test,
        predictions
    )
)

# SAVE MODEL

model.save(
    "CNN_GW_Merger_Detector.h5"
)

print("Model Saved Successfully")